## Computing cleanup stats

#### Global Rejection rates

The firs section is kept only for historical purpose, as i think it is an erronous way of computing global values.

In [1]:
import platform
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, HTML, display_html

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella, utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()
global_wf = dataset_utils.open_wf(dataset.meta.lang_word_frequency("EN"))
global_rejected_wf = dataset_utils.open_wf(dataset.meta.lang_rejected_word_frequency("EN"))
global_preprocess_wf = dataset_utils.open_wf(dataset.meta.lang_preprocessed_word_frequency("EN"))

data = [
    {"set": "RAW", "tokens": global_preprocess_wf["freq"].sum(), "types": len(global_preprocess_wf["word"])},
    {"set": "Rejected", "tokens": global_rejected_wf["freq"].sum(), "types": len(global_rejected_wf["word"])},
    {"set": "Clean", "tokens": global_wf["freq"].sum(), "types": len(global_wf["word"])},
]
display_html(HTML("<h2> Global STATS (Bad Calculation)</h2>"))
df = pd.DataFrame(data)
display(df.style.format(
    {
        "tokens": "{:,}",
        "types": "{:,}",
    }
))

print(f"""
Token rejection rate across the STELATranscription/EN dataset is {global_rejected_wf["freq"].sum() / global_preprocess_wf["freq"].sum():.2%}
Type  rejection rate across the STELATranscription/EN dataset is {len(global_rejected_wf["word"]) / len(global_preprocess_wf["word"]):.2%}
""")

Global STATS (Bad Calculation)

,set,tokens,types
0,RAW,"294,109,819","336,025"
1,Rejected,"1,647,394","255,770"
2,Clean,"289,664,394","132,201"



Token rejection rate across the STELATranscription/EN dataset is 0.56%
Type  rejection rate across the STELATranscription/EN dataset is 76.12%



##### Correct Rejectation Rate Computation

The correct way to compute a more global image is to make compute averages for each block.

In [2]:
import platform
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella, utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()


@dataclass
class STAT:
    token_rejection_rate: float
    tokens: int
    types: int
    type_rejection_rate: float


global_stats = {}
with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    for lang in dataset.languages:
        for hour in dataset.hour_splits:
            hour_stats = []
            for item in dataset.iter_split(lang, hour):
                try:
                    cleaned_wf = dataset_utils.open_wf(item.clean.word_frequencies)
                    rejected_wf = dataset_utils.open_wf(item.rejected.word_frequencies)
                    preprocess_wf = dataset_utils.open_wf(item.preprocess.word_frequencies)
                    stats = STAT(
                        tokens=cleaned_wf["freq"].sum(),
                        types=len(cleaned_wf["word"]),
                        token_rejection_rate=rejected_wf["freq"].sum() / preprocess_wf["freq"].sum(),
                        type_rejection_rate=len(rejected_wf["word"]) / len(preprocess_wf["word"]),
                    )
                except Exception:
                    print(f"Failed for {lang}/{hour}/{section}")
                    raise
            hour_stats.append(stats)
            global_stats[f"{lang}/{hour}"] = STAT(
                tokens=np.nanmean([s.tokens for s in hour_stats]),
                types=np.nanmean([s.types for s in hour_stats]),
                token_rejection_rate=np.nanmean([s.token_rejection_rate for s in hour_stats]),
                type_rejection_rate=np.nanmean([s.type_rejection_rate for s in hour_stats])
            )


stela_simple_word_cleaning_stats_df = pd.DataFrame([{"section": k, **asdict(v)} for k, v in global_stats.items()])
stela_simple_word_cleaning_stats_df.columns = [
    "section", "Token RJ (Avg per block)", "Tokens (Avg per block)", "Types (Avg per block)", "Type RJ (Avg per block)"
]
%store stela_simple_word_cleaning_stats_df

Output()

Succesfully computed all word frequencies ! (Total time: 22 seconds)

Stored 'stela_simple_word_cleaning_stats_df' (DataFrame)


In [3]:
from IPython.display import display, HTML, display_html
display_html(HTML("<h2> Global Average STATS</h2>"))
display_html(HTML("<p> Average on each block inside each hour split. EN/50h is the result of the average of al EN/50h/XX blocks.<p>"))
display(stela_simple_word_cleaning_stats_df.style.format(
    {
        "Token RJ (Avg per block)": "{:.2%}",
        "Tokens (Avg per block)": lambda x: '{:,}'.format(int(x)),
        "Types (Avg per block)": lambda x: '{:,}'.format(int(x)),
        "Type RJ (Avg per block)": "{:.2%}",
    }
))

Global Average STATS

Average on each block inside each hour split. EN/50h is the result of the average of al EN/50h/XX blocks.

,section,Token RJ (Avg per block),Tokens (Avg per block),Types (Avg per block),Type RJ (Avg per block)
0,EN/50h,0.59%,"570,279","20,403",3.36%
1,EN/100h,0.64%,"1,060,532","29,267",6.04%
2,EN/200h,0.67%,"2,358,527","42,324",11.52%
3,EN/400h,0.78%,"4,655,191","58,253",22.97%
4,EN/800h,1.00%,"8,810,671","75,258",35.16%
5,EN/1600h,1.71%,"20,108,147","102,994",48.26%
6,EN/3200h,1.51%,"41,395,136","132,201",60.66%


# Progressive Word cleaning using chunk-average

In [4]:
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.stats import block_average2
from lexical_benchmark.utils import timed_status

loading_status = "Compiling CHUNKs & extracting rates..."
complete_status = "Completed computation of all rates !"
dataset = stella.STELATranscriptDataset()
en_dict = dataset_utils.DictionairyCleaner(lang="EN")

WORDS_3200h_00 = dataset.item("EN", "3200h", "00").preprocess.processed.read_tokenized()
CHUNK_SPLITS = [
    1600,
    3200,
    6400,
    12_800,
    25_600,
    51_200,
    102_400,
    204_800,
    409_600,
    819_200,             # ~50h
    1_638_400,           # ~100h
    3_276_800,           # ~200h
    6_553_600,           # ~400h
    13_107_200,          # ~800h
    26_214_400,          # ~1600h
    len(WORDS_3200h_00)  # ~3200h
]



def word_clean_fn(word: str) -> bool:
    """Check if a word is in dict"""
    global en_dict
    return en_dict.check(word)


stats = {}
with timed_status(status=loading_status, complete_status=complete_status):
    for chunk_size in CHUNK_SPLITS:
        chunk_list = block_average2.chunk_splitter(WORDS_3200h_00, chunk_size=chunk_size)
        word_chunk_stats = block_average2.clean_chunk_list(chunks=chunk_list, filter_fn=word_clean_fn)
        stats[chunk_size] = word_chunk_stats

Output()

Completed computation of all rates ! (Total time: 7 minutes and 30 seconds)

In [5]:
from IPython.display import display
import pandas as pd

rj_rate = [
    {
        "split": label,
        "Token Rejection Rate": cs.rejection_rate,
        "Type Rejection Rate": cs.unique_rejection_rate,
        "Total Words": cs.total_words,
    }
    for label, cs in stats.items()
]
df = pd.DataFrame(rj_rate)
display(df.style.format({
    "split": lambda x: f"{int(x):,}",
    "Token Rejection Rate": "{:.3%}",
    "Type Rejection Rate": "{:.3%}",
    "Total Words": lambda x: f"{int(x):,}"
}))
!date

,split,Token Rejection Rate,Type Rejection Rate,Total Words
0,"1,600",1.511%,2.924%,"42,028,800"
1,"3,200",1.511%,3.292%,"42,028,800"
2,"6,400",1.511%,3.734%,"42,028,800"
3,"12,800",1.511%,4.271%,"42,022,400"
4,"25,600",1.511%,4.937%,"42,009,600"
5,"51,200",1.511%,5.795%,"41,984,000"
6,"102,400",1.511%,6.948%,"41,984,000"
7,"204,800",1.511%,8.492%,"41,984,000"
8,"409,600",1.509%,10.692%,"41,779,200"
9,"819,200",1.509%,14.152%,"41,779,200"


Fri Dec  6 03:51:41 PM CET 2024
